# Encoders and Heads

We studied a lot of models in the last session when trying to treat text as a sequence of tokens. 

Let's try to generalize them a bit. 
Generally, we have two parts of a model.


### First Model

```py
class MeanPoolClf(nn.Module):

    def __init__(self, n_words=tok.vocab_size, n_dim=100, pad_id=tok.pad_token_id):
        super().__init__()
        self.embeddings = nn.Embedding(n_words, n_dim, padding_idx=pad_id)
        self.classifier = nn.Linear(n_dim, 1)

    def forward(self, input_ids, attention_mask):
        H = self.embeddings(input_ids)                 # (B, T, D)
        context = masked_mean(H, attention_mask)       # (B, D)
        return self.classifier(context).squeeze(-1)    # (B,)
```

Let's represent it like this:

![](../resources/imgs/6_3_recap1.png)


Let's make it a bit abstract

![](../resources/imgs/6_3_recap1_1.png)

We can see it as having rougly two parts:
1. **Encoder**: Refine the representation of inputs to be richer
2. **Head**: Take the refined input and do something with it

![](../resources/imgs/6_3_recap_encdec.png)

-----

### Second Model

```py
class MeanPoolMLPClf(nn.Module):

    def __init__(self, n_words=tok.vocab_size, n_dim=100, hidden=100, pad_id=tok.pad_token_id):
        super().__init__()
        self.embeddings = nn.Embedding(n_words, n_dim, padding_idx=pad_id)
        self.head = nn.Sequential(
            nn.Linear(n_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, input_ids, attention_mask):
        H = self.embeddings(input_ids)                 # (B, T, D)
        context = masked_mean(H, attention_mask)       # (B, D)
        return self.head(context).squeeze(-1)          # (B,)
```

We can also express this very similary. **Only the classifier head got more complicated. The encoder was the same as #1.**

![](../resources/imgs/6_3_recap2.png)


----

### Fourth Model

```py
class TokenFFNClf(nn.Module):

    def __init__(self, n_words=tok.vocab_size, n_dim=100, hidden=100, pad_id=tok.pad_token_id):
        super().__init__()
        self.embeddings = nn.Embedding(n_words, n_dim, padding_idx=pad_id)
        
        self.block = nn.Sequential(               # an MLP, applied to each token separately
            nn.Linear(n_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_dim),             # note: back to n_dim, so we can stack these
        )
        self.classifier = nn.Linear(n_dim, 1)

    def encode_tokens(self, input_ids):
        """(B, T) -> (B, T, D). Kept separate so we can look inside later."""
        return self.block(self.embeddings(input_ids))

    def forward(self, input_ids, attention_mask):
        H = self.encode_tokens(input_ids)              # (B, T, D)

        # Flatten time. Bye bye sequentiality
        context = masked_mean(H, attention_mask)       # (B, D)
        return self.classifier(context).squeeze(-1)    # (B,)
```

Here we **expanded the encoder** but kept the classifier head same as in #1

![](../resources/imgs/6_3_recap3_1.png)

OR

![](../resources/imgs/6_3_recap3_2.png)


## Same in images

We had a similar set up going in our image classifier.

![](../resources/imgs/6_3_recap_conv.png)


**It is a common pattern in deep learning models**

# Today

We will focus on the encoder, and create richer representation of text using **self attention**

In [ ]:
# tqdm.std = plain-text bars. tqdm.auto would use ipywidgets here, whose
# comm channels wedge the VSCode notebook connection after a long loop.
from tqdm.std import tqdm, trange
from datasets import load_dataset
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
# Check if you have nvidia GPU, you may be able to use cuda
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

In [ ]:
# dataset = load_dataset("glue", "cola")
dataset = load_dataset("stanfordnlp/sst2")

# NOTE: SST-2's *test* split has hidden labels (all -1) because it's a GLUE leaderboard task.
#       So we use `validation` as our test set. This is standard practice for SST-2.
train = dataset["train"]
test  = dataset["validation"]

len(train), len(test), train[1]

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tok.tokenize("the movie was not good"))
print("vocab size:", tok.vocab_size, "| pad id:", tok.pad_token_id)

In [ ]:
MAXLEN = 64   # SST-2 sentences are short; this truncates almost nothing

def encode(texts, maxlen=MAXLEN):
    out = tok(texts, padding="max_length", truncation=True,
              max_length=maxlen, return_tensors="pt")
    return out["input_ids"], out["attention_mask"]

train_ids, train_mask = encode(train["sentence"])
test_ids,  test_mask  = encode(test["sentence"])
train_y = torch.tensor(train["label"], dtype=torch.float)
test_y  = torch.tensor(test["label"],  dtype=torch.float)

train_ids.shape, train_mask.shape, train_y.shape

In [ ]:
train_loader = DataLoader(
    TensorDataset(train_ids.to(device), train_mask.to(device), train_y.to(device)),
    batch_size=256, shuffle=True)
test_loader = DataLoader(
    TensorDataset(test_ids.to(device), test_mask.to(device), test_y.to(device)),
    batch_size=256)

for batch in train_loader:
    break
[t.shape for t in batch]

In [ ]:
lossfn = nn.BCEWithLogitsLoss()

def accuracy(logits, y):
    return ((logits > 0).float() == y).float().mean().item()

@torch.no_grad()
def evaluate(model):
    model.eval()
    accs = [accuracy(model(ids, mask), y) for ids, mask, y in test_loader]
    return sum(accs) / len(accs)


def report(model):
    """Print a model's entrails: what parameters exist, how big, and which ones learn."""
    total = trainable = 0
    print(f"{'parameter':<32}{'shape':<18}{'count':>12}   learns?")
    print("-" * 72)
    for name, prm in model.named_parameters():
        total += prm.numel()
        trainable += prm.numel() if prm.requires_grad else 0
        print(f"{name:<32}{str(tuple(prm.shape)):<18}{prm.numel():>12,}   "
              f"{'yes' if prm.requires_grad else 'FROZEN'}")
    print("-" * 72)
    print(f"{'':<32}{'TOTAL':<18}{total:>12,}   ({trainable:,} learn)")


def plot_history(history, name=""):
    """Left: loss at every batch. Right: per-epoch loss against test accuracy."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), dpi=110)

    steps = history["batch_loss"]
    ax[0].plot(steps, alpha=0.25, lw=0.7, label="each batch")
    w = max(1, len(steps) // 100)                       # smooth through the noise
    if w > 1:
        smooth = np.convolve(steps, np.ones(w) / w, mode="valid")
        ax[0].plot(range(w - 1, len(steps)), smooth, lw=1.8, label=f"moving avg ({w})")
    ax[0].set_xlabel("batch"); ax[0].set_ylabel("loss")
    ax[0].set_title("loss, every single batch"); ax[0].legend()

    ep = range(1, len(history["epoch_loss"]) + 1)
    ax[1].plot(ep, history["epoch_loss"], "o-", color="tab:blue")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("avg loss", color="tab:blue")
    ax[1].set_title("per epoch"); ax[1].set_xticks(list(ep))
    acc_ax = ax[1].twinx()
    acc_ax.plot(ep, history["test_acc"], "s--", color="tab:red")
    acc_ax.set_ylabel("test accuracy", color="tab:red")

    fig.suptitle(name or "training", y=1.04)
    plt.tight_layout(); plt.show()


def train_model(model, epochs=3, lr=1e-3, name="", plot=True):
    """Trains `model` IN PLACE. Returns the loss/accuracy history."""
    name = name or model.__class__.__name__
    model.to(device)                                    # nn.Module.to() is in place
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"batch_loss": [], "epoch_loss": [], "test_acc": []}

    epoch_bar = trange(epochs, desc=name)
    for e in epoch_bar:
        model.train()
        running = []
        batch_bar = tqdm(train_loader, leave=False, desc=f"epoch {e}", mininterval=0.5)
        for ids, mask, y in batch_bar:
            opt.zero_grad()
            loss = lossfn(model(ids, mask), y)
            loss.backward()
            opt.step()

            running.append(loss.item())
            history["batch_loss"].append(loss.item())

        acc = evaluate(model)
        history["epoch_loss"].append(float(np.mean(running)))
        history["test_acc"].append(acc)
        epoch_bar.set_postfix(loss=f"{history['epoch_loss'][-1]:.4f}", test_acc=f"{acc:.3f}")
        tqdm.write(f"epoch {e}: avg loss = {history['epoch_loss'][-1]:.4f}  "
                   f"test acc = {acc:.3f}")

    if plot:
        plot_history(history, name)
    return history

In [ ]:
# Last week's pooling, carried over untouched. Note where it is about to live: in the HEAD.
def masked_mean(H, attention_mask):
    """H: (B, T, D), attention_mask: (B, T) of 0/1  ->  (B, D)"""
    m = attention_mask.unsqueeze(-1).float()          # (B, T, 1)
    return (H * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)


# Let's isolate the head

`Classifier` takes an **encoder** and puts a decision on top of it. That's the whole thing:
pool with a masked mean, then one `Linear` to a single logit.

Let's just abstract it out

In [ ]:
# LIVE: the body of forward. Two lines, and both are last week's.
class Classifier(nn.Module):
    """The head.   (B,T,D) + mask  ->  (B,)

    Pool the token vectors down to one, then one Linear to a single logit. That is all a
    head is. We write it once today; every model below keeps one of these, unchanged.
    """

    def __init__(self, n_dim=100):
        super().__init__()
        self.out = nn.Linear(n_dim, 1)

    def forward(self, H, attention_mask):
        context = masked_mean(H, attention_mask)      # (B,D)   <- the lossy step
        return self.out(context).squeeze(-1)          # (B,)


In [ ]:
# Our two instruments. Both from last week; both take a model and print one thing.

@torch.no_grad()
def order_probe(model, pos="not bad but good", neg="not good but bad"):
    """Same words, different order, opposite meaning. Does the LOGIT move at all?"""
    ids, mask = encode([pos, neg])
    logits = model(ids.to(device), mask.to(device))
    d = (logits[0] - logits[1]).mean()
    print(f"{pos!r} -> {logits[0]:>9.6f} | {neg!r} -> {logits[1]:>9.6f} | diff {d:.2e}")
    return d


@torch.no_grad()
def sequence_probe(model, a="the movie was not good", b="the movie was not great",
                   n=None, plot=True):
    """Change ONE word. Which token vectors move, and by how much?

    Note this calls `encoder`, not the model — we are looking at the ENCODER's
    output, (B,T,D), before the head has pooled anything away.
    """
    ids_a, mask_a = encode([a])
    ids_b, mask_b = encode([b])
    Ha = model.encoder(ids_a.to(device), mask_a.to(device))
    Hb = model.encoder(ids_b.to(device), mask_b.to(device))

    # Calculate unpadded lengths using attention masks (or non-pad token counts)
    if n is None:
        len_a = mask_a[0].sum().item() if mask_a is not None else (ids_a[0] != tok.pad_token_id).sum().item()
        len_b = mask_b[0].sum().item() if mask_b is not None else (ids_b[0] != tok.pad_token_id).sum().item()
        n = min(len_a, len_b)

    toks  = tok.convert_ids_to_tokens(ids_a[0][:n])
    delta = (Ha[:, :n] - Hb[:, :n]).abs().sum(dim=-1)[0]
    for t, d in zip(toks, delta):
        print(f"  {d.item():>10.4f}   {t}")

    # Diff word
    words_a = set(a.split())
    words_b = set(b.split())
    if len(words_b-words_a) == 1:
        title = f"What changed when {(words_a-words_b).__iter__().__next__()} became {((words_b-words_a).__iter__().__next__())}"
    else:
        title = "What happens to encoder outputs"

    if plot:
        plt.figure(figsize=(10, 2.8))
        plt.imshow((Ha[:, :n] - Hb[:, :n]).abs()[0].detach().cpu(), aspect='auto', cmap='coolwarm')
        plt.yticks(range(n), toks)
        plt.xlabel("dimension"); plt.title(title)
        plt.colorbar(); plt.show()
    return delta

# Approach0: Last session's approach

Last week's Attempt 1, in the new shape. 
Encoder:
    - just an embedding layer
Head:
    - the one above

In [ ]:
class MeanPoolClf(nn.Module):
    """Last week's Attempt 1, with the head split out into its own object.

    Encoder: a lookup table. ZERO layers — nothing is transformed, ids go in, rows come out.
    Head: the Classifier we just wrote.
    """

    def __init__(self, n_words=tok.vocab_size, n_dim=100, pad_id=tok.pad_token_id):
        super().__init__()
        ...
        self.classifier = Classifier(n_dim)

    def encoder(self, input_ids, attention_mask=None):
        """(B,T) -> (B,T,D). """
        ...

    def forward(self, input_ids, attention_mask):
        ...


In [ ]:
a0 = MeanPoolClf()
report(a0)          # 3 million params, and 3 million of them are the lookup table

hist_e0 = train_model(a0, name="A0: embeddings only", epochs=10)


# E1: last week's best encoder

Attempt 4 from 6.2 — a position-wise MLP. Same weights applied to every token, independently.
The genuinely new thing last session; a recap this session.

Note what stays put: `Classifier` is untouched, so the head is the same object it was for E0.

In [ ]:
# LIVE: `encoder` and `forward`.
class TokenFFNClf(nn.Module):
    """Last week's Attempt 4, unchanged, with the head split out.

    Encoder: lookup table, then an MLP applied to each token separately.
    Head:    the same Classifier as E0. Same class, same shape, nothing touched.
    """

    def __init__(self, n_words=tok.vocab_size, n_dim=100, hidden=100, pad_id=tok.pad_token_id):
        super().__init__()
        ...
        self.classifier = Classifier(n_dim)

    def encoder(self, input_ids, attention_mask=None):
        """(B,T) -> (B,T,D)."""
        ...

    def forward(self, input_ids, attention_mask):
        ...

In [ ]:
a1 = TokenFFNClf()
report(a1)          # 20,200 more than E0. That's `block`.

hist_e1 = train_model(a1, name="E1: position-wise FFN", epochs=10)


# This one is worse. Why?

- Both models have one problem
- A0 is not bloated. A1 is more bloated. A0 wins. 

# Problem: One token affects only one embedding

In [ ]:
order_probe(a1)
print('------------')
sequence_probe(a1)

# It'll be the same for A0 btw


Order probe will be zero. Always. Oh well. 

But see what happens to the encoder output when we change one word `good` -> `great`!

- the embedding for good changes. Yay!
- the embedding for all other words don't change at all. Noooo! Sadge!


And so, both encoders only deal with one token in isolation. 
See below:

In [ ]:
sequence_probe(a1, a="I never said she stole my money.", b="I always said she stole my money.")

The embeddings for "said" should change based on prev word being "never" vs "always" right?


## The encoded output needs to be sequence aware


### We kinda did that already.

Our **masked mean** takes every token embedding, and "pools" the information into one vector. 

![](../resources/imgs/6_3_recap_seqaware.png)

It just happens to live in the **head**. It also happens to be *kinda stupid*.
Let's see why
<!-- So what if we put it in the **encoder**? -->

In [ ]:
ids, mask = encode(["the movie was not good"])
n = int(mask[0].sum())
H = a1.embeddings(ids.to(device))                          # (B,T,D)

new_token_vector = masked_mean(H, mask.to(device))                  # (B,D)

print("H      ", tuple(H.shape))
print("summary", tuple(new_token_vector.shape), "<- ONE vector, for a sentence of", n, "tokens")


Now this vector **does represent something from each token**, and is **sequence aware**. 
i.e.

$$h_t = f(\text{sequence})$$

But should we pass the same vector for each token?



`not` and `movie` and `the` would all receive the *identical*
vector. There's no way for `good` to get a summary that leans on `not` in particular.

We need a way to get a mix of current token's inputs, and other tokens' inputs conditioned on the current token, i.e.,

$$h_t = f(\text{sequence}, t)$$


## The encoded output needs to be sequence aware AND token aware

### But first — what *is* a mean?

We are about to change `masked_mean`, so let's be sure we know what it does. It is four
operations and not one of them is clever.


In [ ]:
# let's say for sequence: 
text = "I am"
text2 = "the movie was not good"



# Get ID and mask
ids, mask

In [ ]:
# Get the embeddings


# Hidden encoding of the first doc
h1 = H[1]
h1.mean(dim=0)

### So, in symbols

$$
c \;=\; \frac{1}{T}\sum_{t=1}^{T} h_t
$$



### Now push the division inside the sum

$$
c \;=\; \sum_{t=1}^{T} \frac{1}{T}\, h_t
\;=\; \sum_{t=1}^{T} a_t\, h_t
\qquad\text{where}\qquad a_t = \frac{1}{T}
$$

$w_t$ is a *weight*, one per token, we set them all to be **the same** $\frac{1}{T}$.

In [ ]:
h_avg = ...



**every token in this sentence matters exactly as much as every other token**

# And then each token becomes this average token
(in this faulty thought experiment)



In [ ]:
c1 = ...




# What if the weights were different per token?

> the movie was not good

- so that CLS token may be focused on everyhing
- people largely ignored "the"
- "not" becomes important
... and so on...

The only thing we actually need is for the weights to sum to one — otherwise we're scaling the
vector rather than averaging it:

$$
\sum_{t=1}^{T} a_t = 1
$$

Within that, we can pick anything we like.


![](../resources/imgs/6_3_attnmat.png)


$$
z_i \;=\; \sum_{j=1}^{T} A_{ij}\, h_j
\qquad\text{with}\qquad
\sum_{j=1}^{T} A_{ij} = 1 \;\text{ for every } i
$$

$a$ was a vector of length $T$. $A$ is a $T \times T$ **matrix** — one row per token, and row
$i$ says how much token $i$ weighs each token $j$.

And notice what the plain mean becomes in that language. It is the corner case where every row
is the same, and flat:

$$
A_{ij} = \frac{1}{T} \quad \text{for all } i, j
$$

![](../resources/imgs/6_3_attnmat_uniform.png)

In [ ]:
# The cell above, on the tensors we already have. A is 7x7 and every entry is 1/7.
T = h1.shape[0]                  # 7 tokens, no padding — that is why we picked this sentence
A = torch.full((T, T), 1 / T, device=device)

print("A ="); print(A.cpu().numpy().round(3))
print("row sums:", [round(x, 3) for x in A.sum(dim=1).tolist()])
print()
print("A @ h1  ==  the vstack we built by hand :", torch.allclose(A @ h1, c1, atol=1e-6))
print("output row 0  ==  output row 6          :", torch.allclose((A @ h1)[0], (A @ h1)[6], atol=1e-6))

### The `vstack` was a matrix multiply all along

`c1` was seven hand-written copies of `h_avg`. `A @ h1` is those same seven rows, and we never
said the word "mean" anywhere — we wrote down a matrix and multiplied.

So the pooling we have been using since 6.2 was already $AH$. We were only ever allowed *one*
$A$: the one where every row is identical, and every entry in it is $\tfrac{1}{T}$.

Nothing stops us writing down a different one.

In [ ]:
toks = tok.convert_ids_to_tokens(ids[1])
print("tokens:", list(enumerate(toks)))          # 4 is `not`, 5 is `good`

print()
print("A ="); print(A.cpu().numpy().round(3))

# Leave six rows alone. Give row 5 (`good`) different weights: half `not`, half itself.
A2 = A.clone()
A2[5]    = 0.0
A2[5, 4] = 0.5
A2[5, 5] = 0.5

print()
print("A2 ="); print(A2.cpu().numpy().round(3))
print("row sums:", [round(x, 3) for x in A2.sum(dim=1).tolist()])    # still all 1.0
print('-------------- DIFF --------------')

# Which output vectors moved, and by how much? (Same readout as `sequence_probe`.)
Z, Z2 = A @ h1, A2 @ h1
for t, d in zip(toks, (Z - Z2).abs().sum(dim=-1)):
    print(f"  {d.item():>9.4f}   {t}")

print()
print("`good` came out exactly halfway between `not` and `good`:",
      torch.allclose(Z2[5], 0.5 * h1[4] + 0.5 * h1[5], atol=1e-6))

### One row changed, and only that token moved

Six zeros and one number. We edited row 5, so output vector 5 changed.


In [ ]:
# One more shape check, because this is the shape the model will actually build: (B,T,T).
w = mask.float().to(device) / mask.sum(dim=1, keepdim=True).to(device)   # (B,T)  the 1/T weights
A_batch = w.unsqueeze(1).expand(-1, T, -1)                              # (B,T,T) that row, T times

print("A_batch", tuple(A_batch.shape), "-> one T x T matrix per sentence")
print()
print("doc 0 is 'I am' — 4 real tokens, 3 pads, so the pad COLUMNS are zero:")
print(A_batch[0].cpu().numpy().round(3))
print()
print("A_batch @ H  vs  masked_mean(H, mask), biggest gap anywhere:",
      (A_batch @ H - masked_mean(H, mask.to(device)).unsqueeze(1)).abs().max().item())

# E3: let each token choose its own weights

Row `i` should say *how much token `i` should listen to each other token*. So we need a number
for every (token, token) pair — and the number should be big when token `j` is relevant to
token `i`.

## How?

Use vector similarity

### 1. Do the dot product

In [ ]:
h1 = torch.rand(2, 3)
h1_1, h1_2 = h1

### 2. Weights must sum up to 1



### 2.1 How do we handle masked stuff?

Let's get some real data

In [ ]:
# let's say for sequence: 
text = "I am"
text2 = "the movie was not good"
ids, mask = encode([text, text2], maxlen=7)

H = a1.embeddings(ids.to(device)).to('cpu')


In [ ]:
# We need to ignore the masks when doing the softmax.
## The -inf trick


## Finally, adding the attention on the scores

In [ ]:
# LIVE: `scores` and `encoder`.
class DotAttnClf(nn.Module):
    """Each token gets its OWN summary of the sentence, weighted by how similar
    each other token is to it.

    Head: still the same Classifier. Untouched.
    """

    def __init__(self, n_words=tok.vocab_size, n_dim=100, hidden=100, pad_id=tok.pad_token_id):
        super().__init__()
        ...
        self.classifier = Classifier(n_dim)

    def scores(self, H):
        """(B,T,D) -> (B,T,T). How much should each token listen to each other token?"""
        ...

    def attend(self, scores, H, attention_mask):
        """scores: (B,T,T) raw   H: (B,T,D)   ->   mixed (B,T,D), weights (B,T,T)

        Row i of the weights says how much token i listens to each token. Then row i of the
        output is the weighted sum of ALL the token vectors using those weights.
        """
        ...

    def encoder(self, input_ids, attention_mask=None):
        ...

    def forward(self, input_ids, attention_mask):
        ...
        return self.classifier(H, attention_mask)


### Why attention goes *before* the block

Attention gives each token its context; the MLP then gets to think about a token that already
knows its context. That's the order a real transformer block uses, and it means E1's `block` is
reused character for character.

In [ ]:
a3 = DotAttnClf()
report(a3)          # compare the totals with E1's. Look hard.

hist_a3 = train_model(a3, name="A3: dot-product attention")


### `report` says the parameter count didn't change

E1 and E3 have **exactly** the same number of parameters. Attention added zero. Park that too.

In [ ]:
sequence_probe(a3)


# Every row moved.

Compare that with E1, where six of the seven rows were `0.0000` bit for bit. We changed one
word, and the vectors for `[CLS]`, `the`, `movie`, `was`, `not` and `[SEP]` *all* shifted.

> **Every token now knows what the other tokens are.**

That's the wire that didn't exist. It cost three lines and no parameters.

In [ ]:
# The weights are a (T,T) matrix per sentence, so they're a picture.
@torch.no_grad()
def show_attention(model, sentence, ax=None):
    ids, mask = encode([sentence])
    H = model.embeddings(ids.to(device))
    _, A = attend(model.scores(H), H, mask.to(device))
    n = int(mask[0].sum())
    toks = tok.convert_ids_to_tokens(ids[0])[:n]

    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(A[0, :n, :n].detach().cpu(), cmap="viridis")
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(toks, rotation=90); ax.set_yticklabels(toks)
    ax.set_xlabel("...listens to"); ax.set_ylabel("this token...")
    if own:
        plt.colorbar(im); plt.title(f"{sentence!r}"); plt.tight_layout(); plt.show()
    return im


show_attention(a3, "the movie was not good")


### How to read it

Pick a row — say `not`. Scanning across tells you how much `not` listened to each word when it
built its new vector. The row sums to 1.

It's the mean's `(T,T)` matrix from twenty minutes ago, with the rows finally allowed to differ.

Now: is it any *good*? Two problems, and one of them is visible in that picture.

In [ ]:
# Problem 1. Look at the matrix and its transpose.
H = a3.embeddings(ids.to(device))
S = a3.scores(H)[0, :n, :n]

print("max |S - S^T| =", (S - S.T).abs().max().item())


### Problem 1: the matrix is symmetric, and it has to be

`h_i · h_j` and `h_j · h_i` are the same number. So "how much does `good` care about `not`" and
"how much does `not` care about `good`" are **forced to be equal**. Relevance is mutual, whether
we like it or not, and we don't get a vote.

That's fatal for the thing we actually want. We want `good` to hunt for a negation. But `good`
and `not` aren't similar vectors — similarity is all a dot product can measure — so `good` tends
to listen to `not` *less* than to an unrelated word.

Look at the `not` row in the probe output above. It moved **less than any other real token** —
less than `the`. That's not bad luck; it held on every seed I tried, and it's what a symmetric
similarity score does to a word whose whole job is to modify something unlike itself.

### Problem 2: there is nothing to learn

`report` already told us: zero new parameters. The weights in that picture are dictated entirely
by the geometry of the embedding table. The model cannot decide what counts as relevant — and
remember, we had to hand-tune the embedding scale to stop the whole thing collapsing to the
identity.

Both problems have the same cause. **The comparison itself has no parameters in it.**

In [ ]:
# And the other instrument, on all three, one last time.
for name, m in [("a0  embeddings", a0), ("a1  token FFN ", a1), ("a3  attention ", a3)]:
    print(f"{name}  ", end="")
    order_probe(m)


# Still zero. All three.

`1e-07`, six sessions running. We just built the operation that made modern NLP work, and it
still cannot tell `not bad but good` from `not good but bad`.

And it structurally can't. **Attention treats the sentence as a set.** `score(i, j)` is a
function of two vectors; it knows nothing about where either one sits. Reorder the words and
every row of `A` holds the same numbers in a different order, so every output vector is the
same, so the mean is the same.

Look at `attend` and `scores` again. There is no `t` anywhere in either of them. Nothing in the
entire encoder ever receives a position.


# Where we got to

    input_ids -> embed -> [ attend ] -> [ block ] -> mean -> Linear -> logit
                  (B,T,D)   (B,T,D)      (B,T,D)     (B,D)    (B,1)

Those two middle boxes are a transformer block. Attention mixes the tokens; the MLP then
processes each mixed token. BERT is twelve of them stacked, with a `Linear` on top.

We changed the encoder three times today and never once touched the head:

| encoder | rows that move | attention params | symmetric? |
|---|---|---|---|
| a0  lookup table | 1 of 7 | — | — |
| a1  + position-wise MLP | 1 of 7 | — | — |
| a3  + dot-product attention | **7 of 7** | 0 | yes, unavoidably |

## Two things wrong, and they're both still open

| | what it costs us | when |
|---|---|---|
| **The comparison has no parameters** | symmetric scores, so `not` and `good` are stuck caring about each other equally; and we had to hand-pick the embedding scale to stop attention collapsing to the identity | next session |
| **No positions** | attention is a set operation, so the order probe stays at `1e-07` | the session after |

The first one is one line away, and it is the whole of next session.

## And a word about today's numbers

Attention bought roughly one point of test accuracy over `a1` — measured over three seeds,
`0.8033` against `0.7944`. Real, repeatable, and tiny.

That is not why we did any of this. Everything we actually learned today came from two probes, a
heatmap, and one print of `S - S^T`.


# Stuff to try (optional, obviously)

- Delete the `weight *= 0.1` line from `DotAttnClf`, retrain, and run the sequence probe. You'll
  get `a1`'s answer — one row moves. Attention is still in there, doing nothing at all. Worth
  seeing once.
- Print `a3.embeddings.weight.norm(dim=1).mean()` before and after training. Did the table grow?
  Does the attention picture get sharper?
- Add the paper's `/ H.shape[-1] ** 0.5` inside `scores` and diff the heatmaps.
- Run `attend` twice in `encoder`, before the block. Does the sequence probe change? The order
  probe? (Predict first, then check.)
- Put the attention *after* the block instead of before. Anything measurable? Which order do you
  think a real transformer uses, and why?
- Feed a `(B,T,T)` matrix of your own invention to `attend` — e.g. every token attends only to
  its immediate neighbours. That's a convolution, and it's 5.3 turning up again.
- Swap `masked_mean` in the head for `H[:, 0]` — just the `[CLS]` vector. Last session that was
  a trap because `[CLS]` knew nothing about the sentence. Is it still a trap?
- Hand-write an `A` for `"the movie was not good"` that you think is *correct* — which token
  should listen to which — and push it through `attend` yourself. Then compare it with the one
  the model came up with.


# Further Reading

- *Attention Is All You Need* — https://arxiv.org/abs/1706.03762. Section 3.2.1 is what we built
  today. Equation (1) is our `attend`, with two details we haven't earned yet.
- The Illustrated Transformer — https://jalammar.github.io/illustrated-transformer/ — for the
  pictures. Read it *after* the query/key/value session, not before.
- Andrej Karpathy, *Let's build GPT: from scratch, in code, spelled out* —
  https://www.youtube.com/watch?v=kCc8FmEb1nY. Arrives at the same place by a different road.
